# **Assignment 2: Model Training and Experiment Tracking Overview**

In this assignment, you will train an MNIST classifier in PyTorch and monitor the training with Weights & Biases (W&B).

From Assignment 1 - reusing pieces to validate

### Step 1: Model Training and Experiment Tracking
In this section, we define our dynamic Multi-Layer Perceptron (`LogisticRegression`) architecture, initialize our Weights & Biases (W&B) project, set up the data loaders for the MNIST dataset, and train the model. Metrics such as training loss, validation loss, and accuracy are logged dynamically to W&B.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import wandb

class LogisticRegression(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_layers=None, dropout_rate=0.2):
        super(LogisticRegression, self).__init__()
        if hidden_layers is None:
            hidden_layers = [128, 64]

        layers = []
        prev_dim = input_dim
        for h_dim in hidden_layers:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = h_dim

        layers.append(nn.Linear(prev_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        # Flatten the input images
        x = x.view(x.size(0), -1)
        return self.network(x)

def unnormalize(tensor, mean=0.1307, std=0.3081):
    """Reverses the MNIST normalization so images render correctly."""
    return tensor * std + mean

# Hyperparameters to experiment with
config = {
    "learning_rate": 0.01,
    "epochs": 10,
    "batch_size": 64,
    "hidden_layers": [128, 64],
    "dropout_rate": 0.2
}

# Initialize wandb
wandb.init(project="mnist-mlops", config=config)
config = wandb.config

# Load and preprocess data
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

# Initialize model, loss, and optimizer
model = LogisticRegression(
    input_dim=28*28,
    output_dim=10,
    hidden_layers=config.hidden_layers,
    dropout_rate=config.dropout_rate
)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

# Training loop
for epoch in range(config.epochs):
    model.train()
    epoch_loss = 0.0
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        if batch_idx % 100 == 0:
            wandb.log({"batch_loss": loss.item()})

    avg_train_loss = epoch_loss / len(train_loader)
    wandb.log({"epoch": epoch, "train_loss": avg_train_loss})

    # Validation
    model.eval()
    correct = 0
    total = 0
    val_loss = 0.0
    with torch.no_grad():
        for data, target in test_loader:
            output = model(data)
            loss = criterion(output, target)
            val_loss += loss.item()
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()

    accuracy = 100.0 * correct / total
    avg_val_loss = val_loss / len(test_loader)
    wandb.log({"val_loss": avg_val_loss, "val_accuracy": accuracy})
    print(f"Epoch {epoch+1}/{config.epochs} - Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f} - Val Acc: {accuracy:.2f}%")

# Log final sample validation predictions to W&B as requested
model.eval()
images, labels = next(iter(test_loader))
with torch.no_grad():
    preds = model(images[:16]).argmax(dim=1)

unnormalized_images = unnormalize(images[:16])

wandb.log({
    "val_predictions": [
        wandb.Image(img, caption=f"true={t.item()} pred={q.item()}")
        for img, t, q in zip(unnormalized_images, labels[:16], preds)
    ]
})

# Save the trained model weights
torch.save(model.state_dict(), "model.pt")

wandb.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: r-d-julin (r-d-julin-iowa-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/10 - Loss: 0.4541 - Val Loss: 0.2527 - Val Acc: 92.43%
Epoch 2/10 - Loss: 0.3723 - Val Loss: 0.2253 - Val Acc: 93.69%
Epoch 3/10 - Loss: 0.3488 - Val Loss: 0.2359 - Val Acc: 94.17%
Epoch 4/10 - Loss: 0.3409 - Val Loss: 0.2465 - Val Acc: 93.89%
Epoch 5/10 - Loss: 0.3429 - Val Loss: 0.2593 - Val Acc: 93.47%
Epoch 6/10 - Loss: 0.3229 - Val Loss: 0.2088 - Val Acc: 94.48%
Epoch 7/10 - Loss: 0.3202 - Val Loss: 0.2492 - Val Acc: 94.37%
Epoch 8/10 - Loss: 0.3249 - Val Loss: 0.2426 - Val Acc: 94.27%
Epoch 9/10 - Loss: 0.3193 - Val Loss: 0.2348 - Val Acc: 94.54%
Epoch 10/10 - Loss: 0.3068 - Val Loss: 0.2348 - Val Acc: 94.42%


batch_loss,█▂▂▁▂▁▂▂▂▁▁▂▁▂▂▂▂▂▂▂▂▁▁▂▂▂▁▂▁▂▁▂▁▁▂▂▂▂▂▃
epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▄▃▃▃▂▂▂▂▁
val_accuracy,▁▅▇▆▄█▇▇██
val_loss,▇▃▅▆█▁▇▆▅▅
batch_loss,0.72356
epoch,9
train_loss,0.30685
val_accuracy,94.42
val_loss,0.2348


### Step 2: Post-Training Evaluation and Gradient Auditing
Here, we initialize a dedicated W&B evaluation run to log supplementary validation metrics—specifically calculating the gradient norms to ensure stable learning, checking the learning rate, and evaluating precision and recall scores. We also visualize and log actual prediction images to the dashboard.

In [3]:
# Initialize a new run to log post-training validation analysis
wandb.init(project="mnist-mlops", job_type="evaluation")

# Watch the model
wandb.watch(model, log="all")

# Helper to calculate gradient norm
def get_grad_norm(model):
    total_norm = 0.0
    for p in model.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item() ** 2
    return total_norm ** 0.5

# Log details (gradient norm, learning rate)
current_lr = optimizer.param_groups[0]['lr']
grad_norm = get_grad_norm(model)
wandb.log({
    "learning_rate": current_lr,
    "gradient_norm": grad_norm
})

# After validation, log more metrics (precision and recall)
model.eval()
all_preds = []
all_targets = []
with torch.no_grad():
    for data, target in test_loader:
        output = model(data)
        _, predicted = torch.max(output, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(target.cpu().numpy())

import numpy as np
from sklearn.metrics import precision_recall_fscore_support

precision, recall, _, _ = precision_recall_fscore_support(
    all_targets, all_preds, average='macro', zero_division=0
)

wandb.log({
    "precision_macro": precision,
    "recall_macro": recall
})

# Helper function to un-normalize MNIST images back to [0, 1] range
def unnormalize(tensor, mean=0.1307, std=0.3081):
    return tensor * std + mean

# Visualize sample predictions with un-normalization implemented
def log_predictions(model, data, target, num_samples=10):
    model.eval()
    with torch.no_grad():
        output = model(data)
        _, preds = torch.max(output, 1)

    # Un-normalize the batch of images first
    unnormalized_data = unnormalize(data)

    images = []
    for i in range(min(num_samples, len(data))):
        img = unnormalized_data[i].squeeze().cpu().numpy()
        pred = preds[i].item()
        truth = target[i].item()
        images.append(wandb.Image(
            img,
            caption=f"Pred: {pred}, Truth: {truth}"
        ))
    wandb.log({"sample_predictions": images})

# Call this function to visualize predictions
log_predictions(model, next(iter(test_loader))[0][:10], next(iter(test_loader))[1][:10])

wandb.finish()

gradient_norm,▁
learning_rate,▁
precision_macro,▁
recall_macro,▁
gradient_norm,0.58035
learning_rate,0.01
precision_macro,0.9442
recall_macro,0.9435


### Step 3: Hyperparameter Experimentation
To optimize our network, we run comparative experiments utilizing different model topologies (e.g., shallow-wide vs. deep network architectures), varying learning rates, and different dropout rates. These metrics are logged across separate runs so they can be compared in the W&B dashboard, complete with confusion matrices.

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import wandb
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

# Helper function to un-normalize MNIST images back to [0, 1] range
def unnormalize(tensor, mean=0.1307, std=0.3081):
    return tensor * std + mean

# Prepare data loaders
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Define configurations to experiment with
experiments = [
    {
        "model_name": "Deeper_MLP",
        "hidden_layers": [256, 128, 64],
        "learning_rate": 0.001,
        "dropout_rate": 0.3,
        "batch_size": 128,
        "epochs": 5
    },
    {
        "model_name": "Shallow_Wide_MLP",
        "hidden_layers": [512],
        "learning_rate": 0.0005,
        "dropout_rate": 0.1,
        "batch_size": 64,
        "epochs": 5
    }
]

for run_idx, config in enumerate(experiments):
    print(f"\n--- Starting Experiment: {config['model_name']} ---")

    # Init W&B run
    run = wandb.init(
        project="mnist-mlops",
        name=f"run-{config['model_name']}",
        config=config,
        reinit=True
    )

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=config["batch_size"], shuffle=False)

    # Use the dynamic MLP architecture from previous steps
    model = LogisticRegression(
        input_dim=28*28,
        output_dim=10,
        hidden_layers=config["hidden_layers"],
        dropout_rate=config["dropout_rate"]
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config["learning_rate"])

    # Train
    for epoch in range(config["epochs"]):
        model.train()
        epoch_loss = 0.0
        for data, target in train_loader:
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_train_loss = epoch_loss / len(train_loader)

        # Evaluate
        model.eval()
        correct = 0
        total = 0
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for data, target in test_loader:
                output = model(data)
                _, predicted = torch.max(output, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()
                all_preds.extend(predicted.cpu().numpy())
                all_targets.extend(target.cpu().numpy())

        accuracy = 100.0 * correct / total
        precision, recall, _, _ = precision_recall_fscore_support(
            all_targets, all_preds, average='macro', zero_division=0
        )

        # Log standard epoch metrics
        wandb.log({
            "epoch": epoch,
            "train_loss": avg_train_loss,
            "val_accuracy": accuracy,
            "precision_macro": precision,
            "recall_macro": recall
        })
        print(f"Epoch {epoch+1}/{config['epochs']} - Train Loss: {avg_train_loss:.4f} - Val Accuracy: {accuracy:.2f}%")

    # Log a final Confusion Matrix to W&B dashboard
    wandb.log({
        "confusion_matrix": wandb.plot.confusion_matrix(
            probs=None,
            y_true=all_targets,
            preds=all_preds,
            class_names=[str(i) for i in range(10)]
        )
    })

    # Visual Prediction Samples with Correct Un-normalization
    model.eval()
    images, labels = next(iter(test_loader))
    with torch.no_grad():
        preds = model(images[:10]).argmax(dim=1)

    unnormalized_images = unnormalize(images[:10])

    wandb.log({
        "sample_predictions": [
            wandb.Image(img.squeeze().cpu().numpy(), caption=f"Pred: {p.item()}, Truth: {t.item()}")
            for img, p, t in zip(unnormalized_images, preds, labels[:10])
        ]
    })

    run.finish()


--- Starting Experiment: Deeper_MLP ---


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 1/5 - Train Loss: 0.4405 - Val Accuracy: 95.45%
Epoch 2/5 - Train Loss: 0.1905 - Val Accuracy: 96.60%
Epoch 3/5 - Train Loss: 0.1484 - Val Accuracy: 97.17%
Epoch 4/5 - Train Loss: 0.1243 - Val Accuracy: 97.58%
Epoch 5/5 - Train Loss: 0.1093 - Val Accuracy: 97.61%


epoch,▁▃▅▆█
precision_macro,▁▅▇██
recall_macro,▁▅▇██
train_loss,█▃▂▁▁
val_accuracy,▁▅▇██
epoch,4
precision_macro,0.97591
recall_macro,0.97602
train_loss,0.10931
val_accuracy,97.61



--- Starting Experiment: Shallow_Wide_MLP ---


Epoch 1/5 - Train Loss: 0.2484 - Val Accuracy: 96.48%
Epoch 2/5 - Train Loss: 0.1044 - Val Accuracy: 97.15%
Epoch 3/5 - Train Loss: 0.0691 - Val Accuracy: 97.50%
Epoch 4/5 - Train Loss: 0.0522 - Val Accuracy: 97.66%
Epoch 5/5 - Train Loss: 0.0400 - Val Accuracy: 98.12%


epoch,▁▃▅▆█
precision_macro,▁▄▅▆█
recall_macro,▁▄▅▆█
train_loss,█▃▂▁▁
val_accuracy,▁▄▅▆█
epoch,4
precision_macro,0.98126
recall_macro,0.981
train_loss,0.04004
val_accuracy,98.12


### Step 4: Model Serving with FastAPI
Once training is complete, we load our saved weights (`model.pt`) into the model architecture and configure a REST API using FastAPI. We configure a `/predict/` endpoint that accepts uploaded images, handles preprocessing (grayscale, resize, normalization), performs model inference, and returns predictions asynchronously.

In [5]:
from fastapi import FastAPI, File, UploadFile, HTTPException
from PIL import Image
import torch
import torchvision.transforms as transforms
import io
import nest_asyncio
import uvicorn
import asyncio

app = FastAPI(
    title="MNIST MLOps Inference Service",
    description="FastAPI server with robust error handling for MNIST digit predictions"
)

# Configure model structure with weights loaded
try:
    model = LogisticRegression(input_dim=28*28, output_dim=10, hidden_layers=[128, 64], dropout_rate=0.2)
    model.load_state_dict(torch.load("model.pt", map_location=torch.device("cpu")))
    model.eval()
except Exception as e:
    raise RuntimeError(f"Failed to load PyTorch model weights: {str(e)}")

@app.post("/predict/")
async def predict(file: UploadFile = File(...)):
    # 1. Input Validation: Check file type extension
    if not file.content_type.startswith("image/"):
        raise HTTPException(
            status_code=400,
            detail=f"Invalid file type: '{file.content_type}'. Please upload an image file (PNG/JPEG)."
        )

    # Read file content safely
    try:
        content = await file.read()
        if not content:
            raise HTTPException(status_code=400, detail="Uploaded file is empty.")
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Failed to read upload stream: {str(e)}")

    # 2. Image Processing & Conversion Validation
    try:
        image = Image.open(io.BytesIO(content)).convert('L')
    except Exception:
        raise HTTPException(
            status_code=400,
            detail="Uploaded file is not a valid or readable image. Verification failed."
        )

    # 3. Model Inference Preprocessing
    try:
        transform = transforms.Compose([
            transforms.Resize((28, 28)),
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])
        tensor = transform(image).unsqueeze(0)
    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"Error during image preprocessing: {str(e)}"
        )

    # 4. Perform Inference Safely
    try:
        with torch.no_grad():
            outputs = model(tensor)
            prediction = outputs.argmax(dim=1).item()
    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"Inference engine failure: {str(e)}"
        )

    return {
        "success": True,
        "prediction": int(prediction)
    }

if __name__ == "__main__":
    # Apply nest_asyncio to prevent loop collisions inside Google Colab
    nest_asyncio.apply()
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
    server = uvicorn.Server(config)
    loop = asyncio.get_event_loop()
    loop.create_task(server.serve())


### Step 5: Containerization with Docker
To package the model and application for robust production deployments, we write a `Dockerfile`. This configuration specifies a lightweight Python base, installs essential requirements (including CPU-only PyTorch to minimize image size), copies the saved `model.pt` weights, and sets up FastAPI as the entrypoint instruction.

In [6]:
%%writefile Dockerfile
# FIXME: Choose an appropriate base image
FROM python:3.10-slim

WORKDIR /app

# FIXME: Copy necessary files and install dependencies
# We need packages like torch (cpu-only to keep size smaller), torchvision, fastapi, uvicorn, nest_asyncio, and Pillow.
RUN pip install --no-cache-dir fastapi uvicorn Pillow nest_asyncio \
    --extra-index-url https://download.pytorch.org/whl/cpu \
    torch torchvision

# Copy our model weights and our application code
COPY model.pt /app/model.pt
# In a typical production setup, the Python application source would be copied as well
# COPY app.py /app/app.py

# EXPOSE the port FastAPI app is configured to run on
EXPOSE 8000

# FIXME: Set up the command to run your FastAPI application
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

Overwriting Dockerfile


### Summary of the Dockerfile Build Process

The written `Dockerfile` acts as a step-by-step recipe to containerize our MNIST FastAPI application. Here is what happens during the image build process:

1. **Base Image Selection (`FROM`):** We use `python:3.10-slim` as a lightweight Debian-based Python base image to keep the overall container size small.
2. **Workspace Setup (`WORKDIR`):** We set `/app` as the active directory inside the container for all subsequent copying and running operations.
3. **Dependencies Installation (`RUN`):** We install `fastapi`, `uvicorn`, `Pillow`, and `nest_asyncio`. Critically, we retrieve CPU-only versions of `torch` and `torchvision` using PyTorch's custom index URL (`https://download.pytorch.org/whl/cpu`) to avoid downloading heavy, unnecessary GPU-accelerated libraries.
4. **Application Files (`COPY`):** We copy the pre-trained weights (`model.pt`) into `/app` so the API can use them for local inference.
5. **Port Configuration (`EXPOSE`):** We expose port `8000` to allow traffic to flow into our server.
6. **Container Startup (`CMD`):** The default execution command launches the FastAPI application using `uvicorn` listening on port `8000` across all interfaces (`0.0.0.0`).

### Step 6: Automated CI/CD Workflow with GitHub Actions
Finally, we write a GitHub Actions workflow YAML configuration file (`.github/workflows/deploy.yml`). This automates our integration and deployment pipeline: checking out code, setting up Python, running test suites, building/pushing the production Docker container to Docker Hub, and deploying the image to Heroku.

In [7]:
%%writefile .github/workflows/deploy.yml
name: Deploy FastAPI App

on:
  push:
    branches: [ main ]

jobs:
  deploy:
    runs-on: ubuntu-latest
    steps:
    - uses: actions/checkout@v4
    - name: Set up Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.10'
    - name: Install dependencies
      run: |
        python -m pip install --upgrade pip
        pip install -r requirements.txt
    - name: Run tests
      run: pytest
    # FIXME: Add steps to build and push Docker image
    - name: Log in to Docker Hub
      uses: docker/login-action@v3
      with:
        username: ${{ secrets.DOCKERHUB_USERNAME }}
        password: ${{ secrets.DOCKERHUB_TOKEN }}
    - name: Build and push Docker image
      uses: docker/build-push-action@v5
      with:
        context: .
        push: true
        tags: ${{ secrets.DOCKERHUB_USERNAME }}/mnist-fastapi:latest

    # FIXME: Add steps to deploy your application (e.g., to Heroku or AWS)
    - name: Deploy to Heroku
      env:
        HEROKU_API_KEY: ${{ secrets.HEROKU_API_KEY }}
        HEROKU_APP_NAME: "mnist-fastapi-app"
      run: |
        docker tag ${{ secrets.DOCKERHUB_USERNAME }}/mnist-fastapi:latest registry.heroku.com/$HEROKU_APP_NAME/web
        docker login --username=_ --password=$HEROKU_API_KEY registry.heroku.com
        docker push registry.heroku.com/$HEROKU_APP_NAME/web
        heroku container:release web --app $HEROKU_APP_NAME

Overwriting .github/workflows/deploy.yml


Checking to validate the test file deploymnt worked

In [8]:
%%writefile test_deployment.py
import pytest
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
import io
from fastapi import FastAPI, File, UploadFile
import torchvision.transforms as transforms
from fastapi.testclient import TestClient

# Redefining the LogisticRegression model inline for self-contained testing
class LogisticRegression(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_layers=None, dropout_rate=0.2):
        super(LogisticRegression, self).__init__()
        if hidden_layers is None:
            hidden_layers = [128, 64]

        layers = []
        prev_dim = input_dim
        for h_dim in hidden_layers:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = h_dim

        layers.append(nn.Linear(prev_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.network(x)

# Instantiate a test app inline matching the production FastAPI environment
app = FastAPI()
model = LogisticRegression(input_dim=28*28, output_dim=10, hidden_layers=[128, 64], dropout_rate=0.2)
# Safe mock loading if no local weights found
try:
    model.load_state_dict(torch.load("model.pt", map_location=torch.device("cpu")))
except Exception:
    pass
model.eval()

@app.post("/predict/")
async def predict(file: UploadFile = File(...)):
    image = Image.open(io.BytesIO(await file.read())).convert('L')
    transform = transforms.Compose([
        transforms.Resize((28, 28)),
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    tensor = transform(image).unsqueeze(0)
    with torch.no_grad():
        outputs = model(tensor)
        prediction = outputs.argmax(dim=1).item()
    return {"prediction": int(prediction)}

def test_model_inference():
    """Verify that our model handles correct input shape and generates valid logit dimensions."""
    test_model = LogisticRegression(input_dim=28*28, output_dim=10, hidden_layers=[128, 64], dropout_rate=0.2)
    dummy_input = torch.randn(2, 1, 28, 28)
    output = test_model(dummy_input)
    assert output.shape == (2, 10)

def test_api_predict():
    """Verify that the FastAPI predict endpoint correctly processes uploaded image files."""
    client = TestClient(app)
    img = Image.new('L', (28, 28), color=0)
    img_byte_arr = io.BytesIO()
    img.save(img_byte_arr, format='PNG')
    img_byte_arr.seek(0)

    response = client.post(
        "/predict/",
        files={"file": ("test_digit.png", img_byte_arr, "image/png")}
    )

    assert response.status_code == 200
    json_data = response.json()
    assert "prediction" in json_data
    assert isinstance(json_data["prediction"], int)
    assert 0 <= json_data["prediction"] <= 9


Overwriting test_deployment.py


In [9]:
# Install necessary testing dependencies and run the updated pytest suite locally
!pip install -q httpx pytest
!pytest test_deployment.py

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0
rootdir: /content
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 2 items                                                              

test_deployment.py ..                                                    [100%]

============================== 2 passed in 3.76s ===============================
